Transformers.ipynb

Author: Dr. Jorge Luis Rosas Trigueros

Last modification: 25abr26

https://tinyurl.com/mt4uz35v

In [1]:
# 1. Update pip and install the Unsloth engine
!pip install --upgrade pip -q
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# 2. Explicitly install xformers and other dependencies
# Note: xformers is now added to the list. We use the --index-url to ensure
# we get the pre-compiled 2026 binary for Colab's CUDA version.
!pip install --no-deps xformers trl peft accelerate bitsandbytes unsloth_zoo -q

# 3. Suppress the chatter
import warnings
import logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

# print("✅ Installation complete! xformers is now present.")

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # None for auto detection
load_in_4bit = True # Use 4bit to fit in free Colab

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/tinyllama-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Define the Alpaca-style prompt template
alpaca_prompt = """### Instruction:
{}

### Context:
{}

### Response:
{}"""

In [ ]:
# 1. Prepare the prompt with a 'nudge' (a trailing space)
test_instruction = "Complete the following thought: The most important thing about learning AI is"
test_context = ""

prompt = alpaca_prompt.format(test_instruction, test_context, "")
# We strip trailing whitespace and add exactly one space to 'prime' the engine
prompt = prompt.rstrip() + " "

# 2. Tokenize
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 3. Generate with 'min_new_tokens' to force it past the empty response
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

print("--- FORCED GENERATION ---")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128,
    min_new_tokens = 5,      # <--- FORCES the model to speak
    temperature = 0.5,
    do_sample = True,
    repetition_penalty = 1.2 # Prevents it from getting stuck
)

In [4]:
# 1. Prepare the prompt with a 'nudge' (a trailing space)
test_instruction = "Complete the following thought: The most important thing about learning AI is"
test_context = ""

prompt = alpaca_prompt.format(test_instruction, test_context, "")
# We strip trailing whitespace and add exactly one space to 'prime' the engine
prompt = prompt.rstrip() + " "

# 2. Tokenize
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 3. Generate with 'min_new_tokens' to force it past the empty response
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

print("--- FORCED GENERATION ---")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128,
    min_new_tokens = 5,      # <--- FORCES the model to speak
    temperature = 0.5,
    do_sample = True,
    repetition_penalty = 1.2 # Prevents it from getting stuck
)

In [5]:
# 1. Prepare the prompt with a 'nudge' (a trailing space)
test_instruction = "Complete the following thought: The most important thing about learning AI is"
test_context = ""

prompt = alpaca_prompt.format(test_instruction, test_context, "")
# We strip trailing whitespace and add exactly one space to 'prime' the engine
prompt = prompt.rstrip() + " "

# 2. Tokenize
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 3. Generate with 'min_new_tokens' to force it past the empty response
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

print("--- FORCED GENERATION ---")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128,
    min_new_tokens = 5,      # <--- FORCES the model to speak
    temperature = 0.5,
    do_sample = True,
    repetition_penalty = 1.2 # Prevents it from getting stuck
)

In [6]:
# 1. Prepare the prompt with a 'nudge' (a trailing space)
test_instruction = "What is the specific length of the Great Wall of China?"
test_context = ""

prompt = alpaca_prompt.format(test_instruction, test_context, "")
# We strip trailing whitespace and add exactly one space to 'prime' the engine
prompt = prompt.rstrip() + " "

# 2. Tokenize
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 3. Generate with 'min_new_tokens' to force it past the empty response
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

print("--- FORCED GENERATION ---")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128,
    min_new_tokens = 5,      # <--- FORCES the model to speak
    temperature = 0.5,
    do_sample = True,
    repetition_penalty = 1.2 # Prevents it from getting stuck
)

In [7]:
from datasets import load_dataset

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    contexts     = examples["context"]
    outputs      = examples["response"]
    texts = []
    for instruction, context, output in zip(instructions, contexts, outputs):
        text = alpaca_prompt.format(instruction, context, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

# Load only the first 500 rows for a quick lab session
dataset = load_dataset("databricks/databricks-dolly-15k", split = "train[:500]")
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [8]:
if tokenizer is None:
    print("Error: Tokenizer is not defined. Re-run Step 2!")
else:
    print(f"Tokenizer is ready: {tokenizer.name_or_path}")

In [9]:
# Check for peft_config to see if the model is already a LoRA model
if not hasattr(model, "peft_config"):
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
    )
    print("✅ LoRA adapters successfully added!")
else:
    # This block now runs silently or with your custom message
    # and prevents calling the Unsloth function that prints the "funny" message.
    pass

In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 200, # enough for a noticeable change
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

trainer.train()

In [11]:
# 1. Prepare the prompt with a 'nudge' (a trailing space)
test_instruction = "Complete the following thought: The most important thing about learning AI is"
test_context = ""

prompt = alpaca_prompt.format(test_instruction, test_context, "")
# We strip trailing whitespace and add exactly one space to 'prime' the engine
prompt = prompt.rstrip() + " "

# 2. Tokenize
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 3. Generate with 'min_new_tokens' to force it past the empty response
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

print("--- FORCED GENERATION ---")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128,
    min_new_tokens = 5,      # <--- FORCES the model to speak
    temperature = 0.5,
    do_sample = True,
    repetition_penalty = 1.2 # Prevents it from getting stuck
)

In [12]:
# 1. Prepare the prompt with a 'nudge' (a trailing space)
test_instruction = "The author of 'The origin of species' is:"
test_context = ""

prompt = alpaca_prompt.format(test_instruction, test_context, "")
# We strip trailing whitespace and add exactly one space to 'prime' the engine
prompt = prompt.rstrip() + " "

# 2. Tokenize
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 3. Generate with 'min_new_tokens' to force it past the empty response
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

print("--- FORCED GENERATION ---")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128,
    min_new_tokens = 5,      # <--- FORCES the model to speak
    temperature = 0.5,
    do_sample = True,
    repetition_penalty = 1.2 # Prevents it from getting stuck
)

In [13]:
# 1. Prepare the prompt with a 'nudge' (a trailing space)
test_instruction = "Classify these items as 'Electronic' or 'Furniture': Sofa, Laptop, Desk, Smartphone, Bookshelf, Tablet."
test_context = ""

prompt = alpaca_prompt.format(test_instruction, test_context, "")
# We strip trailing whitespace and add exactly one space to 'prime' the engine
prompt = prompt.rstrip() + " "

# 2. Tokenize
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 3. Generate with 'min_new_tokens' to force it past the empty response
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

print("--- FORCED GENERATION ---")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128,
    min_new_tokens = 5,      # <--- FORCES the model to speak
    temperature = 0.5,
    do_sample = True,
    repetition_penalty = 1.2 # Prevents it from getting stuck
)

In [14]:
# 1. Prepare the prompt with a 'nudge' (a trailing space)
test_instruction = "What is the specific length of the Great Wall of China?"
test_context = ""

prompt = alpaca_prompt.format(test_instruction, test_context, "")
# We strip trailing whitespace and add exactly one space to 'prime' the engine
prompt = prompt.rstrip() + " "

# 2. Tokenize
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 3. Generate with 'min_new_tokens' to force it past the empty response
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

print("--- FORCED GENERATION ---")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128,
    min_new_tokens = 5,      # <--- FORCES the model to speak
    temperature = 0.5,
    do_sample = True,
    repetition_penalty = 1.2 # Prevents it from getting stuck
)